# NFT参照コード(mint/burnつき)の使用例

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

動作確認に使用するユーザをロードします。

In [2]:
var users = [];
for(var i=0; i<=5; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u58185320 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u10676071 eEYRycGm4znXxnf7bTngX72JmYk57r
user5: u00571239 eWW6LCfRSpoVuudzZHa4ZwCjkkizov


ユーザ名の逆引き関数を準備します。

In [3]:
var userids = users.map(u => u.id);
function nameOfUserId(id){
    var i = userids.indexOf(id);
    if(i >= 0) return 'user'+i;
    return 'user_not_found';
}

## NFTコレクションのデプロイ

NFTコントラクトのサンプルコードを読み出します。このサンプルコードはNFTのmint/burnを実装しています。

In [4]:
var { default: func } = await import('./nft200.mjs');
var m = /function.*\{([\s\S]*)\}/.exec(func.toString());
var code = m[1];

NFTコレクションの名称と略号を書き換えます。

In [5]:
var code = code.replace(/NFT_NAME = '.*'/, `NFT_NAME = 'sample NFT collection #201'`);
var code = code.replace(/NFT_SYMBOL = '.*'/, `NFT_SYMBOL = 'SMP201'`);

NFTコレクションの管理ユーザをユーザ0に書き換えます。  
管理ユーザはNFTのmintとburnを行うことができます。

In [6]:
var code = code.replace(/ADMIN_USER = '.*'/, `ADMIN_USER = '${users[0].id}'`);

NFTの転送履歴の記録台帳となるスマートコントラクトをデプロイします。  
このスマートコントラクトはログを記録するだけの目的で、実行時の処理はありません。  
変数nftlogにスマートコントラクトのIDが格納されます。

In [7]:
var nftlog = await deploySmartContract({from:'string', to:'string', tokenId:'string'}, function(){}, { name: 'nftlog' });
await rpc.call(adminWallet, 'c1update', {id:nftlog, prop:'mask', value: {dlg:true}});

{
  txno: 702338,
  txid: 'xn9ex8d79B65Wxv6t6L6YUFWmwN7T78eWNGWnzhKF547v',
  status: 'ok',
  value: null
}

上で作成したスマートコントラクトを、NFTコレクションに指定します。  

In [8]:
var code = code.replace(/LOG_CONTRACT = '.*'/, `LOG_CONTRACT = '${nftlog}'`);

NFTコレクションのスマートコントラクトをデプロイします。  
変数nftidにNFTコントラクトのIDが格納されます。

In [9]:
var func = new Function(code);
var nftid = await deploySmartContract({func:'string', args:'any'}, func, { name: 'nft200' });

NFTコントラクトのアクセス範囲を、動作確認に使用するユーザに設定します。

In [10]:
await rpc.call(adminWallet, 'c1update', {id:nftid, prop:'accessible_to', value: userids});

{
  txno: 702342,
  txid: 'xBkz3n6dhEQLFFALkXPHXMEkoMXmFFCvo89aeAG7wQgNz',
  status: 'ok',
  value: null
}

NFTの転送履歴が記録できるようにアクセス権を設定します。  
NFTの転送履歴を全体に開示します。

In [11]:
await rpc.call(adminWallet, 'c1update', {id:nftlog, prop:'add accessible_to', value: [nftid]});
await rpc.call(adminWallet, 'c1update', {id:nftlog, prop:'disclosed_to', value: ['all']});

{
  txno: 702344,
  txid: 'xUnmPFik3YpJ34F3uBP6jLk7oDtQcJaBEGTmPZLniXDvAB',
  status: 'ok',
  value: null
}

## NFTコントラクトの主要なインタフェースの動作確認

### name
NFTコレクションの名称を取得します。

In [12]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'name'});
console.log('name:', resp);

name: {
  txno: 702345,
  txid: 'xauQrBB7SAUurXc9w2qQdwARGZXTvaM3hFco5FDVkWRjDB',
  status: 'ok',
  value: 'sample NFT collection #201'
}


### symbol
NFTコレクションの略号を取得します。

In [13]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'symbol'});
console.log('symbol:', resp);

symbol: {
  txno: 702346,
  txid: 'xefnxX2j2yVBuphTRoEpabm54M8MSoEcUPnwYCyiophP9',
  status: 'ok',
  value: 'SMP201'
}


### totalSupply
NFTコレクションに含まれるトークンの総数を取得します。初期状態では0が返ります。

In [14]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'totalSupply'});
console.log('totalSupply:', resp);

totalSupply: {
  txno: 702347,
  txid: 'xSDGZ3ynNnL4923BN4xcRfWiXw8upVp2rdteExopsiccy',
  status: 'ok',
  value: 0
}


### mint
あらたにトークンを作成します。管理ユーザがオーナーとURIを指定して作成します。  
作成されたトークンのtokenIdが返ります。

In [15]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'mint', args: {to: users[1].id, uri: 'https://somewhere.com/token/A'}});
console.log('created:', resp);

created: {
  txno: 702348,
  txid: 'xLfj4LVeFGpcewYPkv4T9DqtyXetpB8HZWCuTG8qPcgkr',
  status: 'ok',
  value: '1'
}


totalSupplyが1になったことを確認します。

In [16]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'totalSupply'});
console.log('totalSupply:', resp);

totalSupply: {
  txno: 702350,
  txid: 'x9wJXb7gnBawQ9y7yxGgWvKtBKjPd7S6kV6xd3ANnhXQt',
  status: 'ok',
  value: 1
}


さらにいくつかのトークンをmintします。

In [17]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'mint', args: {to: users[1].id, uri: 'https://somewhere.com/token/B'}});
console.log('created tokenId:', resp.value);
var resp = await rpc.call(users[0].wallet, nftid, {func: 'mint', args: {to: users[1].id, uri: 'https://somewhere.com/token/C'}});
console.log('created tokenId:', resp.value);
var resp = await rpc.call(users[0].wallet, nftid, {func: 'mint', args: {to: users[2].id, uri: 'https://somewhere.com/token/D'}});
console.log('created tokenId:', resp.value);
var resp = await rpc.call(users[0].wallet, nftid, {func: 'mint', args: {to: users[3].id, uri: 'https://somewhere.com/token/E'}});
console.log('created tokenId:', resp.value);
var resp = await rpc.call(users[0].wallet, nftid, {func: 'mint', args: {to: users[4].id, uri: 'https://somewhere.com/token/F'}});
console.log('created tokenId:', resp.value);

created tokenId: 2
created tokenId: 3
created tokenId: 4
created tokenId: 5
created tokenId: 6


### balanceOf
指定したオーナが現在保持するトークンの数を取得します。


In [18]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', resp.value);
var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', resp.value);

balanceOf(user1): 3
balanceOf(user2): 1


### ownerOf
指定したトークンの現在のオーナを取得します。

In [19]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'ownerOf', args: {tokenId: '1'}});
console.log('ownerOf(1):', resp.value, nameOfUserId(resp.value));

ownerOf(1): u28169743 user1


### tokenURI
指定したトークンに関するメタデータのURIを取得します。
mintしたときに指定したURIが返ります。

In [20]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenURI', args: {tokenId: '1'}});
console.log('tokenURI(1):', resp);

tokenURI(1): {
  txno: 702364,
  txid: 'xWWFt2yMNNEXNFJUPCWpAYDe2MwSCAcgoMkhUsYhNM2JBB',
  status: 'ok',
  value: 'https://somewhere.com/token/A'
}


### tokenByIndex
NFTコレクションに含まれる全トークンのうち、インデックスで指定されるトークンを取得します。

In [21]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenByIndex', args: {index: 3}});
console.log('tokenByIndex(0):', resp);

tokenByIndex(0): {
  txno: 702365,
  txid: 'xHRzkgM9odZzTBZxBMFkCsedmMWLDNvVT7rbHEwFsXXrPB',
  status: 'ok',
  value: '4'
}


### tokenOfOwnerByIndex
指定されたオーナが保持するトークンのうち、インデックスで指定されたトークンを取得します。

In [22]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenOfOwnerByIndex', args: {owner: users[2].id, index: 0}});
console.log('tokenOfOwnerByIndex(0):', resp);

tokenOfOwnerByIndex(0): {
  txno: 702366,
  txid: 'xHSdwVWpUp4KGtZxUVGAwPDQrSmSRiY7XA3pKxF6Zvxb5',
  status: 'ok',
  value: '4'
}


### トークンの保持状況の確認

In [23]:
for(var i=0; i<=5; i++){
    var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[i].id}}, {readmode: 'fast'});
    var tokens = [];
    var n = resp.value;
    for(var j=0; j<n; j++){
        var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenOfOwnerByIndex', args: {owner: users[i].id, index: j}}, {readmode: 'fast'});
        tokens[j] = resp.value;
    }
    console.log(`tokens of user${i}:`, tokens);
}

tokens of user0: []
tokens of user1: [ '1', '2', '3' ]
tokens of user2: [ '4' ]
tokens of user3: [ '5' ]
tokens of user4: [ '6' ]
tokens of user5: []


### transferFrom
指定したトークンを別のオーナに転送します。（オーナが変わります）  
tokenId=1のトークンをuser1からuser2へ転送します。

In [24]:
var resp = await rpc.call(users[1].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[1].id,
      to: users[2].id,
      tokenId: '1'
    }
});
console.log('transferFrom(user1,user2):', resp);

transferFrom(user1,user2): {
  txno: 702367,
  txid: 'xHARnkqpKHUazYtZwPW2onz6feiXSfCvpAfVY82rfE6Bs',
  status: 'ok',
  value: null
}


転送後の保持数を確認します。

In [25]:
var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[1].id}});
console.log('balanceOf(user1):', resp.value);
var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[2].id}});
console.log('balanceOf(user2):', resp.value);

balanceOf(user1): 2
balanceOf(user2): 2


転送後のオーナを確認します。

In [26]:
var resp = await rpc.call(users[0].wallet, nftid, {func: 'ownerOf', args: {tokenId: '1'}});
console.log('ownerOf(1):', resp.value, nameOfUserId(resp.value));

ownerOf(1): u53371386 user2


さらに、tokenId=1のトークンをuser2からuser3へ転送します。

In [27]:
var resp = await rpc.call(users[2].wallet, nftid, {func: "transferFrom",
    args: {
      from: users[2].id,
      to: users[3].id,
      tokenId: '1'
    }
});
console.log('transferFrom(user2,user3):', resp);

transferFrom(user2,user3): {
  txno: 702372,
  txid: 'xo4rVAmvnZSZ52cK5CbtHxoUgJeAYr844qrWo8FvJ8VSr',
  status: 'ok',
  value: null
}


### トークンの保持状況の確認

In [28]:
for(var i=0; i<=5; i++){
    var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[i].id}}, {readmode: 'fast'});
    var tokens = [];
    var n = resp.value;
    for(var j=0; j<n; j++){
        var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenOfOwnerByIndex', args: {owner: users[i].id, index: j}}, {readmode: 'fast'});
        tokens[j] = resp.value;
    }
    console.log(`tokens of user${i}:`, tokens);
}

tokens of user0: []
tokens of user1: [ '3', '2' ]
tokens of user2: [ '4' ]
tokens of user3: [ '5', '1' ]
tokens of user4: [ '6' ]
tokens of user5: []


### burn  
トークンを削除します  
トークンを削除できるのは、トークンのオーナーまたは管理ユーザです。  


In [29]:
var resp = await rpc.call(users[1].wallet, nftid, {func: 'burn', args: {tokenId: '2'}});
console.log('burn(2) by owner:', resp);
var resp = await rpc.call(users[0].wallet, nftid, {func: 'burn', args: {tokenId: '5'}});
console.log('burn(5) by administrator:', resp);

burn(2) by owner: {
  txno: 702374,
  txid: 'xdyzCvHasQkeDShdorqMZF3z3SuwtgVSKmxiJTG4hvSACB',
  status: 'ok',
  value: null
}
burn(5) by administrator: {
  txno: 702376,
  txid: 'xZJaNxjQTiyjgVgBu4NLEWu6ceTRwYsdYioMqexVdgGqq',
  status: 'ok',
  value: null
}


### トークンの保持状況の確認

In [30]:
for(var i=0; i<=5; i++){
    var resp = await rpc.call(users[5].wallet, nftid, {func: 'balanceOf', args: {owner: users[i].id}}, {readmode: 'fast'});
    var tokens = [];
    var n = resp.value;
    for(var j=0; j<n; j++){
        var resp = await rpc.call(users[5].wallet, nftid, {func: 'tokenOfOwnerByIndex', args: {owner: users[i].id, index: j}}, {readmode: 'fast'});
        tokens[j] = resp.value;
    }
    console.log(`tokens of user${i}:`, tokens);
}

tokens of user0: []
tokens of user1: [ '3' ]
tokens of user2: [ '4' ]
tokens of user3: [ '1' ]
tokens of user4: [ '6' ]
tokens of user5: []


## NFTの転送の履歴の取得
NFTの転送の履歴は、履歴を記録するための専用のコントラクト(LOG_CONTRACTに指定したコントラクト)に記録されています。  
c1queryを使って、転送の履歴を取得することができます。 
ここでは、tokenIdが'1'のNFTの転送の履歴に絞り込んで取得します。 


In [31]:
var resp = await rpc.call(users[5].wallet, 'c1query', {
    type: 'transactions', 
    caller: nftid, 
    callee: nftlog, 
    status: 'ok', 
    arg: ['tokenId', '==', '1'], 
    details: ['argstr']
});
console.log(resp.value.list);

[
  {
    txno: 702373,
    caller: [ 'c088629155', 'nft200@d93319890' ],
    callee: [ 'c020438771', 'nftlog@d93319890' ],
    status: 'ok',
    time: 1753421748457,
    argstr: '{"from":"u53371386","to":"u58185320","tokenId":"1"}'
  },
  {
    txno: 702368,
    caller: [ 'c088629155', 'nft200@d93319890' ],
    callee: [ 'c020438771', 'nftlog@d93319890' ],
    status: 'ok',
    time: 1753421746399,
    argstr: '{"from":"u28169743","to":"u53371386","tokenId":"1"}'
  },
  {
    txno: 702349,
    caller: [ 'c088629155', 'nft200@d93319890' ],
    callee: [ 'c020438771', 'nftlog@d93319890' ],
    status: 'ok',
    time: 1753421738360,
    argstr: '{"from":"","to":"u28169743","tokenId":"1"}'
  }
]
